# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print(len(pop))
pop

4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
names = pop['product_name'].str.lower().str.replace(r'[^a-z]', '', regex=True)
tarts = pop[names.str.contains('poptart')]
straw = tarts[tarts['product_name'].str.lower().str.contains('straw')]
other = tarts.drop(straw.index)
print(straw[['sku', 'product_name', 'category']])
print(other[['sku', 'product_name', 'category']])

             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


**Pop-Tart rows that are a different product:** None, all four pop tart rows are strawberry poptarts and refer to the same product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [8]:
# TODO
n_before = len(cat)
canonical = 'SKU#459812'
mapping = {s: canonical for s in straw['sku'] if s != canonical}
cat['canonical_sku'] = cat['sku'].map(mapping).fillna(cat['sku'])
print(mapping)
cat

{'PT-12': 'SKU#459812', 'POPTART-STRAW': 'SKU#459812', 'PT_STRAWBERRY': 'SKU#459812'}


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku,category_clean,name_key
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,SKU#459812,Breakfast Foods,poptarts strawberry
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,SKU#459812,Breakfast,poptart strawberry
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,SKU#459812,Breakfast,poptarts strawberry
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,SKU#459812,Breakfast,poptarts strawberry
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24,Emergency Supplies,24pk bottled water
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WATER24PK,Emergency,24pack bottled water
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR_24PK,Emergency,24 pack water
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,BATT-100,Emergency Supplies,4pk aa batteries
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BAT-AA4,Emergency Supplies,4pack aa batteries
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BATT_AA_4PK,Emergency,4pack aa batteries


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print(cat['category'].value_counts(dropna=False))
cat['category_clean'] = cat['category'].str.strip().str.title()
print(cat['category_clean'].value_counts(dropna=False))



category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
key = (cat['product_name'].str.lower()
       .str.replace(r'[^a-z0-9 ]', '', regex=True)
       .str.split()
       .apply(lambda w: ' '.join(sorted(w)) if isinstance(w, list) else ''))
cat['name_key'] = key
groups = cat.groupby('name_key').filter(lambda g: g['sku'].nunique() > 1)
groups.sort_values('name_key')[['name_key', 'sku', 'product_name', 'category']]

,name_key,sku,product_name,category
12,24pack beer light,BEER-24,Light Beer (24-pack),Alcoholic Beverages
13,24pack beer light,LTBEER24,Light Beer (24-pack),ALCOHOL
8,4pack aa batteries,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies
9,4pack aa batteries,BATT_AA_4PK,AA Batteries 4-Pack,Emergency
14,board game monopoly,BG-MONO,Monopoly Board Game,ENTERTAINMENT
16,board game monopoly,GAME-MONO,Board Game - Monopoly,ENTERTAINMENT
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,entertainment
23,bread white,BREAD-WH,WHITE BREAD,Grocery
24,bread white,WHITEBREAD,WHITE BREAD,grocery
26,butter peanut,PNUTBTR-16,Peanut Butter,GROCERY


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [9]:
# TODO
assert cat['canonical_sku'].notna().all()
assert len(cat) == n_before
unmapped = ~cat['sku'].isin(mapping.keys())
assert (cat.loc[unmapped, 'canonical_sku'] == cat.loc[unmapped, 'sku']).all()
print('mapping is safe')

mapping is safe


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a: Monopoly has three SKUs (BG-MONO, GAME-MONO, MONOPOLY-01) and AA batteries have three (BAT-AA4, BATT_AA_4PK, BATT-100). I’d group them by normalized name, then check by eye for ones the key misses, like “4pk” vs “4-pack”, and map each group to one SKU.

b: It’s a business decision when the name doesn’t prove two rows are the same product. STREAM-01 (Roku Streaming Stick) and ROKU-STD (Streaming Device) might be the same item or two different devices, so I’d ask the electronics category manager.